In [ ]:
import os
import numpy as np
import pandas as pd
import wfdb
from scipy.interpolate import CubicSpline
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from scipy.signal import find_peaks


#LOAD DATA

before you continue, make sure you've mounted your Google Drive and created your data path first

In [ ]:
datasets = {
    "drivedb" : ("https://physionet.org/files/drivedb/1.0.0/",                   3),
    "nsr2db"  : ("https://physionet.org/files/nsr2db/1.0.0/",                    3),
    "wearable": ("https://physionet.org/files/wearable-exam-stress/1.0.0/data/", 4),
}

for name, (url, cut) in datasets.items():
    dataset_folder = os.path.join(data_path, name)

    if os.path.exists(dataset_folder) and len(os.listdir(dataset_folder)) > 0:
        print(f"{name} sudah ada di Google Drive, skip download")
        continue

    print(f"\n⬇ Download {name}...")
    os.makedirs(dataset_folder, exist_ok=True)

    os.system(
        f"wget -r -N -c -np -nH "
        f"--cut-dirs={cut} "
        f"-P {dataset_folder} "
        f"-e robots=off -q --show-progress {url}"
    )

    print(f"✅ {name} selesai!")

drivedb_path  = os.path.join(data_path, "drivedb")
wearable_path = os.path.join(data_path, "wearable")
nsr_path      = os.path.join(data_path, "nsr2db")

print("\n✅ Semua dataset siap digunakan!")

In [ ]:
print("\n=== WEARABLE ===")
for item in sorted(os.listdir(wearable_path))[:5]:
    full = os.path.join(wearable_path, item)
    print(f"  {'📁' if os.path.isdir(full) else '📄'} {item}")

print("\n=== DRIVEDB ===")
for item in sorted(os.listdir(drivedb_path))[:5]:
    full = os.path.join(drivedb_path, item)
    size = os.path.getsize(full)
    print(f"  📄 {item} ({size:,} bytes)")

print("\n=== NSR2DB ===")
for item in sorted(os.listdir(nsr_path))[:5]:
    full = os.path.join(nsr_path, item)
    size = os.path.getsize(full)
    print(f"  📄 {item} ({size:,} bytes)")

#PATH

In [ ]:
EXAM_FOLDER_MAP = {
    "midterm_1" : "stress_akademik",
    "midterm_2" : "stress_akademik",
    "Final"     : "stress_akademik",
}

DRIVEDB_CONDITION_LABEL = {
    "rest":    "normal",
    "highway": "stress_berkendara",
    "city":    "stress_berkendara",
}

#PREPROCESSING

In [ ]:
#CLEAN RR INTERVAL
def clean_rr(rr_ms, low=300, high=2000):
    rr_ms = np.asarray(rr_ms, dtype=float)
    return rr_ms[(rr_ms > low) & (rr_ms < high)]

#LOAD EXAM STRESS
def load_rr_exam_stress(ibi_filepath, verbose=True):
    df = pd.read_csv(ibi_filepath)

    if df.shape[1] < 2:
        raise ValueError(f"Format IBI.csv tidak sesuai: {ibi_filepath}")

    rr_ms = df.iloc[:, 1].dropna().values.astype(float) * 1000
    rr_ms = clean_rr(rr_ms, 300, 2000)

    print(f"RR min={rr_ms.min():.1f}, max={rr_ms.max():.1f}")

    if verbose:
        print(f"[Exam Stress] {len(rr_ms)} RR intervals loaded dari {os.path.basename(ibi_filepath)}")
        print(f"   RR min={rr_ms.min():.1f} ms | max={rr_ms.max():.1f} ms | mean={rr_ms.mean():.1f} ms")

        if rr_ms.max() > 2000:
            print("WARNING: RR > 2000 ms masih ada!")

    return rr_ms

#LOAD NSR DB
def load_rr_nsr_db(record_path, verbose=True):
    subject_id = os.path.basename(record_path)
    ecg_path   = record_path + ".ecg"

    if not os.path.exists(ecg_path):
        print(f"[NSR DB] SKIP {subject_id}: file .ecg tidak ditemukan")
        return np.array([])

    try:
        raw   = np.fromfile(ecg_path, dtype=np.int16)
        rr_ms = raw[3:].astype(float)
        rr_ms = rr_ms[(rr_ms > 300) & (rr_ms < 2000)]

    except Exception as e:
        print(f"[NSR DB] SKIP {subject_id}: gagal baca → {e}")
        return np.array([])

    if len(rr_ms) == 0:
        print(f"[NSR DB] SKIP {subject_id}: RR kosong setelah filtering")
        return np.array([])

    if verbose:
        print(f"[NSR DB] {subject_id}: {len(rr_ms)} RR | mean={rr_ms.mean():.0f}ms (~{60000/rr_ms.mean():.0f}bpm)")

    return rr_ms

#LOAD DRIVE STRESS
def load_rr_drivedb(record_path, ecg_channel=0, verbose=True):
    subject_id = os.path.basename(record_path)

    try:
        record = wfdb.rdrecord(record_path)
    except Exception as e:
        print(f"[DriveDB] SKIP {subject_id}: gagal baca record → {e}")
        return {}

    fs         = record.fs
    sig_names  = [s.lower() for s in record.sig_name]
    p_signal   = record.p_signal

    if verbose:
        dur = p_signal.shape[0] / fs
        print(f"[DriveDB] {subject_id} | fs={fs}Hz | durasi={dur:.0f}s | channels={record.sig_name}")

    ecg = p_signal[:, ecg_channel]
    ecg = np.where(np.isnan(ecg), np.nanmedian(ecg), ecg)

    # cari marker
    marker_signal = None
    for i, name in enumerate(sig_names):
        if "marker" in name:
            marker_signal = p_signal[:, i]
            break

    if marker_signal is None:
        print(f"[DriveDB] {subject_id}: tidak ada marker channel, SKIP")
        return {}

    # deteksi spike marker
    marker_clean = np.nan_to_num(marker_signal, nan=0.0)
    marker_norm  = (marker_clean - np.min(marker_clean)) / (np.ptp(marker_clean) + 1e-8)

    min_gap_samples = int(300 * fs)
    spike_inds, _ = find_peaks(
        marker_norm,
        prominence=0.3,
        distance=min_gap_samples
    )

    if verbose:
        print(f"[DriveDB] {subject_id}: {len(spike_inds)} marker spike")

    CONDITION_ORDER = ["rest", "highway", "city", "highway", "city", "rest"]

    n_total    = p_signal.shape[0]
    boundaries = [0] + list(spike_inds) + [n_total]

    segments_by_condition = {}

    for seg_idx in range(len(boundaries) - 1):
        if seg_idx >= len(CONDITION_ORDER):
            break

        condition = CONDITION_ORDER[seg_idx]
        start     = boundaries[seg_idx]
        end       = boundaries[seg_idx + 1]

        # filter segmen terlalu pendek (< 5 menit)
        dur_sec = (end - start) / fs
        if dur_sec < 300:
            if verbose:
                print(f"[DriveDB] {subject_id}/{condition}: SKIP durasi {dur_sec:.0f}s < 300s")
            continue

        ecg_seg = ecg[start:end]

        if len(ecg_seg) < int(30 * fs):
            continue

        ecg_norm = (ecg_seg - np.mean(ecg_seg)) / (np.std(ecg_seg) + 1e-8)

        qrs_inds, _ = find_peaks(
            ecg_norm,
            distance=int(0.4 * fs),
            height=0.3,
            prominence=0.3
        )

        if len(qrs_inds) < 2:
            continue

        rr_ms = np.diff(qrs_inds) / fs * 1000
        rr_ms = clean_rr(rr_ms)

        if len(rr_ms) < 20:
            continue

        if condition not in segments_by_condition:
            segments_by_condition[condition] = []

        segments_by_condition[condition].append(rr_ms)

    result = {}
    for condition, rr_list in segments_by_condition.items():
        result[condition] = np.concatenate(rr_list)

    return result


In [ ]:
def correct_outliers(rr_ms, window=11, threshold=0.20, verbose=True):

    rr = np.asarray(rr_ms, dtype=float)
    n = len(rr)

    if n < 5:
        return rr, 0

    window = min(window, n - 1)
    if window % 2 == 0:
        window += 1

    outlier_mask = np.zeros(n, dtype=bool)

    for i in range(n):

        start = max(0, i-window//2)
        end = min(n, i+window//2+1)

        local = np.concatenate([rr[start:i], rr[i+1:end]])

        if len(local) == 0:
            continue

        med = np.median(local)

        if rr[i] < med*(1-threshold) or rr[i] > med*(1+threshold):
            outlier_mask[i] = True

    n_out = outlier_mask.sum()

    if n_out > 0:

        valid = np.where(~outlier_mask)[0]
        bad = np.where(outlier_mask)[0]

        if len(valid) >= 4:
            cs = CubicSpline(valid, rr[valid])
            rr[bad] = cs(bad)
        else:
            rr[bad] = np.median(rr[valid])

    rr = clean_rr(rr,300,2000)

    if verbose:
        print(f"[Outlier] {n_out} titik dikoreksi")
        print(f"   setelah koreksi -> min={rr.min():.1f} max={rr.max():.1f}")

    return rr, int(n_out)

In [ ]:
VALID_LABELS = ("normal", "stress_berkendara", "stress_akademik", None)

def sliding_window_segments(
    rr_ms,
    subject_id=None,
    label=None,
    window_sec=300,
    overlap=0.5,
    min_rr=20
):
    if label not in VALID_LABELS:   # <-- SUDAH DIPERBAIKI
        raise ValueError(f"Label tidak valid: {label}. Gunakan salah satu dari {VALID_LABELS}.")

    rr_ms = np.asarray(rr_ms, dtype=float)

    if len(rr_ms) < min_rr:
        return []

    step_sec = window_sec * (1 - overlap)
    cumtime = np.cumsum(rr_ms) / 1000.0
    total_time = cumtime[-1]

    if total_time < window_sec:
        return []

    segments = []
    start = 0.0

    while start + window_sec <= total_time:
        end = start + window_sec
        mask = (cumtime >= start) & (cumtime < end)
        seg_rr = rr_ms[mask]

        if len(seg_rr) > 0:
            if np.max(seg_rr) > 2000:
                print("=" * 60)
                print("ERROR RR > 2000")
                print("Subject :", subject_id)
                print("Max RR  :", np.max(seg_rr))
                print("RR awal :", seg_rr[:20])
                print("=" * 60)

            if np.isnan(seg_rr).any():
                print(f"NaN ditemukan pada subject {subject_id}")

        if len(seg_rr) >= min_rr:
            segments.append({
                "rr_ms": seg_rr,
                "start_time": round(start, 2),
                "end_time": round(end, 2),
                "subject_id": subject_id,
                "label": label,
            })

        start += step_sec

    return segments

In [ ]:
#HELPER CARI RECORD
def find_records(folder, extension=".hea"):
    records = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.endswith(extension):
                records.append(os.path.join(root, f.replace(extension, "")))
    return sorted(records)

In [ ]:
#LABELING
def label_exam_stress_fixed(base_dir, subject_id):
    all_segs = []

    for folder_name, condition_label in EXAM_FOLDER_MAP.items():
        ibi_path = os.path.join(base_dir, folder_name, "IBI.csv")

        if not os.path.exists(ibi_path):
            print(f"[Exam Stress] File tidak ditemukan: {ibi_path}")
            continue

        rr_ms = load_rr_exam_stress(ibi_path, verbose=True)

        if len(rr_ms) < 20:
            print(f"[Exam Stress] RR terlalu sedikit di {folder_name}, skip")
            continue

        rr_ms, n_out = correct_outliers(rr_ms, verbose=True)

        label = condition_label



        segments = sliding_window_segments(
            rr_ms,
            subject_id=subject_id,
            label=label,
            window_sec=300,
            overlap=0.5
        )

        for seg in segments:
            seg["dataset"]   = "ExamStress"
            seg["condition"] = condition_label

        all_segs.extend(segments)
        print(f"[Exam Stress] {subject_id}/{folder_name}: {len(segments)} segmen (label={label})")

    return all_segs


def label_nsr_db(record_paths, max_seg_per_subject=None):
    all_segs = []

    for record_path in record_paths:
        subject_id = os.path.basename(record_path)

        try:
            rr_ms = load_rr_nsr_db(record_path, verbose=True)
        except Exception as e:
            print(f"[NSR DB] SKIP {subject_id}: gagal load → {e}")
            continue

        if len(rr_ms) < 20:
            print(f"[NSR DB] RR terlalu sedikit di {subject_id}, skip")
            continue

        rr_ms, n_out = correct_outliers(rr_ms, verbose=True)

        segments = sliding_window_segments(
            rr_ms,
            subject_id=subject_id,
            label="normal",
            window_sec=300,
            overlap=0.5
        )

        if max_seg_per_subject and len(segments) > max_seg_per_subject:
            mid      = len(segments) // 2
            half     = max_seg_per_subject // 2
            segments = segments[mid - half : mid + half]

        for seg in segments:
            seg["dataset"]   = "NSR_DB"
            seg["condition"] = "normal_sinus"

        all_segs.extend(segments)
        print(f"[NSR DB] {subject_id}: {len(segments)} segmen (label=normal)")

    return all_segs


def label_drivedb(record_path, verbose=True):
    subject_id = os.path.basename(record_path)

    segments_by_condition = load_rr_drivedb(record_path, verbose=verbose)

    if not segments_by_condition:
        print(f"[DriveDB] SKIP {subject_id}: tidak ada data kondisi")
        return []

    all_segs = []

    for condition, rr_ms in segments_by_condition.items():
        label = DRIVEDB_CONDITION_LABEL.get(condition)


        if label is None:
            continue

        rr_ms, _ = correct_outliers(rr_ms, verbose=verbose)

        segs = sliding_window_segments(
            rr_ms,
            subject_id = subject_id,
            label      = label,
            window_sec = 300,
            overlap    = 0.5
        )

        for seg in segs:
            seg["dataset"]   = "DriveDB"
            seg["condition"] = condition

        all_segs.extend(segs)

        if verbose:
            print(f"[DriveDB] {subject_id}/{condition}: {len(segs)} segmen (label={label})")

    return all_segs

In [ ]:
#MAIN BUAT NGEJALANIN SMUAH
all_segments = []

#Exam Stress
print("=" * 60)
print("PROSES: EXAM STRESS")
print("=" * 60)

exam_subjects = sorted([
    d for d in os.listdir(wearable_path)
    if os.path.isdir(os.path.join(wearable_path, d)) and d.startswith("S")
])
print(f"[Exam Stress] Subjects ditemukan: {exam_subjects}\n")

for subj in exam_subjects:
    base_dir = os.path.join(wearable_path, subj)
    segs     = label_exam_stress_fixed(base_dir, subject_id=subj)
    all_segments.extend(segs)

print(f"\n ExamStress selesai: {len(all_segments)} segmen")

#NSR DB
print("\n" + "=" * 60)
print("PROSES: NSR DB")
print("=" * 60)

nsr_records = find_records(nsr_path)
print(f"[NSR DB] Records ditemukan: {len(nsr_records)}")
print(f"Contoh: {[os.path.basename(r) for r in nsr_records[:3]]}\n")

all_segments.extend(label_nsr_db(nsr_records, max_seg_per_subject=4))  # 3 → 4
print(f"\n NSR DB selesai: {len(all_segments)} segmen total")

#DriveDB
print("\n" + "=" * 60)
print("PROSES: DRIVEDB")
print("=" * 60)

drive_records = find_records(drivedb_path)
print(f"[DriveDB] Records ditemukan: {len(drive_records)}")
print(f"Contoh: {[os.path.basename(r) for r in drive_records[:3]]}\n")

for rp in drive_records:
    segs = label_drivedb(rp, verbose=True)
    all_segments.extend(segs)

print(f"\n DriveDB selesai: {len(all_segments)} segmen total")

print("\nCEK SELURUH SEGMENT")

for i, seg in enumerate(all_segments):

    rr = np.asarray(seg["rr_ms"])

    if np.max(rr) > 2000:

        print("="*60)
        print("SEGMENT ERROR")
        print(i)
        print(seg["dataset"])
        print(seg["subject_id"])
        print(seg["condition"])
        print(rr.max())
        print(rr[:30])

# ── BALANCING: Undersample ke kelas terkecil (3 kelas) ─────────
import random

print("\n" + "=" * 60)
print("BALANCING: UNDERSAMPLE 3 KELAS")
print("=" * 60)

LABELS = ["normal", "stress_berkendara", "stress_akademik"]

groups = {lbl: [s for s in all_segments if s["label"] == lbl] for lbl in LABELS}

for lbl in LABELS:
    print(f"  Sebelum → {lbl}: {len(groups[lbl])}")

min_count = min(len(g) for g in groups.values())
print(f"\n  Undersample semua kelas ke {min_count} segmen (kelas paling sedikit)")

random.seed(42)
all_segments = []
for lbl in LABELS:
    segs = groups[lbl]
    if len(segs) > min_count:
        segs = random.sample(segs, min_count)
        print(f"  {lbl} di-undersample → {len(segs)} segmen")
    all_segments.extend(segs)

random.shuffle(all_segments)
print(f"  Total setelah balancing: {len(all_segments)} segmen")

# CEK CLASS BALANCE
print("\n" + "=" * 60)
print("CEK CLASS BALANCE")
print("=" * 60)

total  = len(all_segments)
counts = {lbl: sum(1 for s in all_segments if s["label"] == lbl) for lbl in LABELS}

for lbl in LABELS:
    c   = counts[lbl]
    pct = c / total * 100 if total else 0
    print(f"  {lbl:20s}: {c} segmen  ({pct:.1f}%)")

if any(c == 0 for c in counts.values()):
    print("Ada kelas yang kosong sama sekali! Cek labeling.")
else:
    ratio = max(counts.values()) / min(counts.values())
    print(f"  Ratio majority:minority = {ratio:.2f}x")
    if ratio > 2.0:
        print("IMBALANCE TERDETEKSI! Pertimbangkan SMOTE atau undersampling.")
    else:
        print("Distribusi label seimbang.")

#EXPORT TO EXCEL

In [ ]:
def segments_to_dataframe(all_segments):
    rows = []
    for i, seg in enumerate(all_segments):
        rows.append({
            "segment_id" : i,
            "subject_id" : seg.get("subject_id", ""),
            "dataset"    : seg.get("dataset", ""),
            "condition"  : seg.get("condition", ""),
            "label"      : seg.get("label", ""),
            "start_time" : seg.get("start_time", 0),
            "end_time"   : seg.get("end_time", 0),
            "n_rr"       : len(seg["rr_ms"]),
            "mean_rr_ms" : round(float(np.mean(seg["rr_ms"])), 2),
            "std_rr_ms"  : round(float(np.std(seg["rr_ms"])),  2),
        })
    return pd.DataFrame(rows)


def save_segments_to_xlsx(all_segments, output_path="segments_labeled.xlsx"):
    df = segments_to_dataframe(all_segments)

    wb = Workbook()

    header_font  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
    header_fill  = PatternFill("solid", start_color="2E4057")
    center_align = Alignment(horizontal="center", vertical="center")
    thin_border  = Border(
        left   = Side(style="thin"),
        right  = Side(style="thin"),
        top    = Side(style="thin"),
        bottom = Side(style="thin")
    )

    LABEL_FILLS = {
        "normal"            : PatternFill("solid", start_color="D6F5D6"),
        "stress_berkendara" : PatternFill("solid", start_color="FFE5B4"),
        "stress_akademik"   : PatternFill("solid", start_color="FFD6D6"),
    }
    default_fill = PatternFill("solid", start_color="EFEFEF")

    def style_header(ws, columns):
        for col_idx, col_name in enumerate(columns, start=1):
            cell           = ws.cell(row=1, column=col_idx, value=col_name)
            cell.font      = header_font
            cell.fill      = header_fill
            cell.alignment = center_align
            cell.border    = thin_border

    #sheet 1: segments
    ws_seg       = wb.active
    ws_seg.title = "Segments"
    ws_seg.row_dimensions[1].height = 20

    columns = [
        "segment_id", "subject_id", "dataset", "condition",
        "label", "start_time", "end_time",
        "n_rr", "mean_rr_ms", "std_rr_ms"
    ]
    style_header(ws_seg, columns)

    for row_idx, row in df.iterrows():
        excel_row = row_idx + 2
        fill = LABEL_FILLS.get(row["label"], default_fill)
        for col_idx, col_name in enumerate(columns, start=1):
            cell           = ws_seg.cell(row=excel_row, column=col_idx, value=row[col_name])
            cell.border    = thin_border
            cell.alignment = Alignment(horizontal="center")
            cell.fill      = fill

    col_widths = [12, 12, 14, 16, 18, 12, 12, 8, 14, 14]
    for col_idx, width in enumerate(col_widths, start=1):
        ws_seg.column_dimensions[
            ws_seg.cell(row=1, column=col_idx).column_letter
        ].width = width

    #sheet 2: summary
    ws_sum = wb.create_sheet("Summary")

    summary_data = (
        df.groupby(["dataset", "label"])
          .size()
          .reset_index(name="n_segments")
    )
    total_per_dataset = (
        df.groupby("dataset")
          .size()
          .reset_index(name="n_segments")
          .assign(label="TOTAL")
    )
    summary_data = pd.concat([summary_data, total_per_dataset], ignore_index=True)
    summary_data = summary_data.sort_values(["dataset", "label"]).reset_index(drop=True)

    sum_cols = ["dataset", "label", "n_segments"]
    style_header(ws_sum, sum_cols)

    total_fill = PatternFill("solid", start_color="FFF2CC")

    for row_idx, row in summary_data.iterrows():
        excel_row = row_idx + 2
        for col_idx, col_name in enumerate(sum_cols, start=1):
            cell           = ws_sum.cell(row=excel_row, column=col_idx, value=row[col_name])
            cell.border    = thin_border
            cell.alignment = Alignment(horizontal="center")
            if row["label"] == "TOTAL":
                cell.fill = total_fill
                cell.font = Font(name="Arial", bold=True)
            else:
                cell.fill = LABEL_FILLS.get(row["label"], default_fill)

    for col_idx, width in enumerate([18, 20, 14], start=1):
        ws_sum.column_dimensions[
            ws_sum.cell(row=1, column=col_idx).column_letter
        ].width = width

    wb.save(output_path)
    print(f"\n Tersimpan: {output_path}")
    print(f"   Total segmen : {len(df)}")
    for lbl in ["normal", "stress_berkendara", "stress_akademik"]:
        print(f"   {lbl:20s}: {(df['label'] == lbl).sum()}")

    return df